# Read vs Write: Measure What One AI Answer Actually Costs

**Time and meter a language model on your own machine, and see where the cost of an answer really goes.**

This is the companion notebook to the [DiamantAI](https://www.youtube.com/@DiamantAI) video **[Why AI Uses So Much Energy (It's Not the Thinking)](https://www.youtube.com/watch?v=w6FgmAtt7oo&list=PLBrpE2PttR2k)**. The film measures one open model on a laptop, reading a whole Sherlock Holmes story and then writing a summary of it. This notebook lets you run the same measurements on yours.

## Overview

A chat answer has two phases. First the model **reads** your prompt (the "prefill"). Then it **writes** its answer one token at a time (the "decode"). They look similar from the outside, but they use the hardware in very different ways, and that difference decides what an answer costs in time and in energy.

This notebook measures it directly with a small open model (Qwen3-0.6B, optionally Qwen3-1.7B) through Hugging Face `transformers`. No API key, and it runs on a laptop CPU, an Apple-silicon Mac or an NVIDIA GPU:

1. **Reading vs writing.** Time the read of a long prompt and the writing of an answer, per token.
2. **Why writing is slow.** Every written token is one full pass over the model's weights, so tokens per second times the model's size is close to your chip's memory bandwidth. We check it on two model sizes.
3. **Long chats.** Grow one conversation's memory (the KV cache) and time each written token as the chat gets longer.
4. **Batching.** Answer 1, 4 and 8 different questions at once and see what each answer costs.
5. **Energy (optional).** If your machine exposes a power sensor, convert time into joules.

On our M4 MacBook Air (Qwen3 8B, 4-bit) we measured: a written token took about **3.5 times the energy** of a read token; each written token cost about **0.095 J per GB of model** the chip had to carry; at 32k tokens of chat each written token cost **2.6 times** what it cost at 1k; and answering **16 questions at once** cost about **half the energy per answer**. Your numbers will differ. The shape should not.

## Detailed Explanation

### Motivation

It is common to hear that an AI answer costs energy because the model "thinks hard". The measurements tell a more specific story. Most of the time and energy goes into moving the model's weights from memory to the compute units, and the amount moved depends on **how many tokens are written**, **how big the model is**, **how long the conversation already is**, and **how many answers share the trip**. Once you can see those four levers, you know which habits actually save something.

### Key Components

- **Read (prefill):** the whole prompt goes through the model in one pass. The weights are loaded once and used for every prompt token in parallel, so the chip is busy doing arithmetic. Cost per token is low.
- **Write (decode):** each new token needs a full pass over all the weights, for just one token of work. The chip spends most of its time waiting on memory. Cost per token is high.
- **KV cache:** the model keeps a key and a value vector for every token seen so far, in every layer. Each written token reads all of it, so a longer chat makes every new token more expensive.
- **Batching:** one pass over the weights can serve many conversations at once. The weight trip is shared, so the cost per answer falls.
- **Power sensor (optional):** NVIDIA's NVML on a GPU, or `powermetrics` on Apple silicon. We subtract the idle draw so the number is the extra energy the answer caused.

### Architecture

![Read vs Write Cost](../images/read_vs_write_cost.svg)

### Benefits

- **Honest numbers from your own hardware**, not from a vendor chart.
- **A physical explanation you can check:** tokens per second times model size should land near your memory bandwidth.
- **Practical habits** that follow directly from the measurements, listed at the end.

## Required Packages

Everything runs locally with PyTorch and `transformers`. A GPU or an Apple-silicon Mac makes it faster, and a CPU also works. No API key is needed. `pynvml` is only for the optional energy step on NVIDIA GPUs.

In [ ]:
!pip install -q torch transformers pandas matplotlib
# optional, NVIDIA only, for the energy step: !pip install -q nvidia-ml-py

## Implementation

### 1. Configuration

The defaults finish in a few minutes on a laptop. `SECOND_MODEL_ID` adds a larger model for step 2; set it to `None` if memory is tight. If the longest chat depth runs out of memory, drop the last entry of `DEPTHS`.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"          # about 1.2 GB of weights in 16-bit
SECOND_MODEL_ID = "Qwen/Qwen3-1.7B"   # optional second size for step 2, or None
READ_TOKENS = 2048                    # how long the prompt is that the model reads
WRITE_TOKENS = 64                     # how many tokens it writes in each timing run
REPEATS = 3                           # every timing runs this many times; we keep the median
DEPTHS = [512, 1024, 2048, 4096, 8192]  # chat lengths for step 3
BATCH_SIZES = [1, 4, 8]               # step 4
BATCH_TOKENS = 64                     # tokens written per answer in step 4

### 2. Load a long text for the model to read

We use *The Adventures of Sherlock Holmes* from Project Gutenberg (public domain), starting at "A Scandal in Bohemia", the story the film uses. It is long enough for every chat depth in step 3.

In [ ]:
import urllib.request

URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
raw = urllib.request.urlopen(URL).read().decode("utf-8")
BOOK = raw[raw.find("I. A SCANDAL IN BOHEMIA"):raw.find("*** END OF")]
print(f"{len(BOOK.split()):,} words")
print(BOOK[:300])

### 3. Load the model and a clock you can trust

We pick CUDA, Apple's MPS or the CPU. GPUs run asynchronously: a Python call returns before the chip has finished. `sync()` waits for the chip, so every timer below measures finished work.

In [ ]:
import gc, statistics, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32

def sync():
    # Wait until the chip has finished everything queued so far.
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(device).eval()
    return tok, model

def weight_bytes(model):
    # Bytes of weights the chip has to read for one full pass (shared weights counted once).
    return sum(p.numel() * p.element_size() for p in model.parameters())

tok, model = load(MODEL_ID)
need = max(DEPTHS + [READ_TOKENS]) + 1024          # tokens we will use, with room to spare
BOOK_IDS = tok(BOOK[:need * 6], return_tensors="pt").input_ids[:, :need].to(device)
print(f"{MODEL_ID} on {device} ({dtype}) | {weight_bytes(model) / 1e9:.2f} GB of weights | "
      f"{BOOK_IDS.shape[1]:,} tokens of text ready")

### 4. The two measurements: read, then write

`read_then_write` does what a chat model does for one answer, with a timer around each phase:

- **Read:** one forward pass over all prompt tokens. `logits_to_keep=1` asks only for the prediction at the last position, like a real server.
- **Write:** a loop that feeds back one token at a time, reusing the KV cache from the read. We pick the most likely token each step (greedy), since the choice of token does not change the cost.

In [ ]:
@torch.no_grad()
def read_then_write(model, ids, n_write):
    # Returns (seconds to read ids, seconds to write n_write tokens, the KV cache).
    sync(); t0 = time.perf_counter()
    out = model(input_ids=ids, use_cache=True, logits_to_keep=1)
    sync(); read_s = time.perf_counter() - t0
    return (read_s, *write(model, out, n_write))

@torch.no_grad()
def write(model, out, n_write):
    # Write n_write tokens one at a time, continuing from a model output that holds a KV cache.
    cache, nxt = out.past_key_values, out.logits[:, -1:].argmax(-1)
    sync(); t0 = time.perf_counter()
    for _ in range(n_write):
        out = model(input_ids=nxt, past_key_values=cache, use_cache=True)
        cache, nxt = out.past_key_values, out.logits[:, -1:].argmax(-1)
    sync()
    return time.perf_counter() - t0, cache

def median_runs(fn, repeats=REPEATS):
    # Run fn once to warm up, then `repeats` times; return the per-field medians.
    fn()
    runs = [fn() for _ in range(repeats)]
    return [statistics.median(r[i] for r in runs) for i in range(len(runs[0]))]

results = {}   # the numbers we collect for the final table

## Usage Example

### Step 1: Reading vs writing, one answer

The model reads `READ_TOKENS` tokens of the story, then writes `WRITE_TOKENS` tokens. We compare the time **per token** in each phase.

In [ ]:
prompt = BOOK_IDS[:, :READ_TOKENS]
read_s, write_s = median_runs(lambda: read_then_write(model, prompt, WRITE_TOKENS)[:2])

ms_read = read_s / READ_TOKENS * 1000
ms_write = write_s / WRITE_TOKENS * 1000
print(f"read  {READ_TOKENS:5d} tokens in {read_s:6.2f} s  ->  {ms_read:6.2f} ms per token read")
print(f"write {WRITE_TOKENS:5d} tokens in {write_s:6.2f} s  ->  {ms_write:6.2f} ms per token written")
print(f"one written token takes as long as {ms_write / ms_read:.0f} read tokens")
results["ms per token read"] = ms_read
results["ms per token written"] = ms_write
results["written / read, time per token"] = ms_write / ms_read

Reading is fast per token because the model loads its weights once and applies them to all prompt tokens together. Writing gets no such sharing: every token pays for its own trip through the weights.

The **time** ratio you see here is larger than the **energy** ratio the film reports (about 3.5 on our Mac). While reading, the chip runs flat out and draws more power; while writing it mostly waits on memory and draws less. Each written token still costs several times the energy of a read one, which is what step 5 measures if you have a power sensor.

## Comparison

### Step 2: Why writing is slow: one pass over the weights per token

If each written token needs the whole model to travel from memory, then

`tokens per second x bytes of weights ≈ bytes per second the memory delivers`

and that product should stay roughly constant across model sizes, near your chip's memory bandwidth (for reference: an M4 MacBook Air is rated at 120 GB/s, an NVIDIA T4 at 320 GB/s). We measure write speed on a short prompt for each model size.

In [ ]:
def write_speed(model, tok):
    # Median tokens per second while writing, after a short prompt.
    ids = tok("Explain how a bicycle stays upright while it moves.", return_tensors="pt").input_ids.to(device)
    _, write_s = median_runs(lambda: read_then_write(model, ids, WRITE_TOKENS)[:2])
    return WRITE_TOKENS / write_s

rows = []
for model_id in [MODEL_ID, SECOND_MODEL_ID]:
    if model_id is None:
        continue
    t, m = (tok, model) if model_id == MODEL_ID else load(model_id)
    tps, gb = write_speed(m, t), weight_bytes(m) / 1e9
    rows.append({"model": model_id.split("/")[-1], "weights GB": gb, "tokens/s written": tps,
                 "GB/s moved (tokens/s x GB)": tps * gb, "ms per written token": 1000 / tps})
    if m is not model:
        del m, t; gc.collect()
        if device == "mps": torch.mps.empty_cache()
        if device == "cuda": torch.cuda.empty_cache()

import pandas as pd
sizes = pd.DataFrame(rows)
results["GB/s moved while writing (largest model)"] = sizes["GB/s moved (tokens/s x GB)"].iloc[-1]
sizes.round(2)

The bigger model writes more slowly, by roughly the ratio of the sizes, and the GB/s column stays in the same range: the chip is limited by how fast it can move weights, not by how much arithmetic it can do. A small model in plain PyTorch often lands below the others, because a fixed per-token overhead (launching hundreds of small GPU kernels) is a bigger share of its short pass. Runtimes built for serving, such as llama.cpp, MLX or vLLM, remove most of that overhead, which is why the film's MLX run reached about 91-103 GB/s on every size.

### Step 3: Long chats: the same model, a growing memory

A chat model keeps a **KV cache**: for every token so far, in every layer, a key vector and a value vector. Every new token reads all of it. We feed the story into one cache, pausing at each depth in `DEPTHS` to time a few written tokens.

In [ ]:
@torch.no_grad()
def write_at_depths(model, ids, depths, n_write=32, chunk=512):
    # Grow one KV cache through `depths` and time n_write written tokens at each depth.
    rows, cache, fed = [], None, 0
    for depth in depths:
        while fed < depth:
            n = min(chunk, depth - fed)
            out = model(input_ids=ids[:, fed:fed + n], past_key_values=cache, use_cache=True, logits_to_keep=1)
            cache, fed = out.past_key_values, fed + n
        length, kv_bytes = cache.get_seq_length(), cache_bytes(cache)
        write(model, out, 4)                      # warm-up at this depth
        times = [write(model, out, n_write)[0] for _ in range(REPEATS)]
        fed = cache.get_seq_length()              # the probe tokens stay in the chat, like a real reply
        rows.append({"chat length (tokens)": length,
                     "ms per written token": statistics.median(times) / n_write * 1000,
                     "KV cache MB": kv_bytes / 1e6})
    return pd.DataFrame(rows)

def cache_bytes(cache):
    # Actual bytes held in a DynamicCache (works across transformers versions).
    pairs = [(l.keys, l.values) for l in cache.layers] if hasattr(cache, "layers") \
        else zip(cache.key_cache, cache.value_cache)
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in pairs)

depth = write_at_depths(model, BOOK_IDS, [d for d in DEPTHS if d <= BOOK_IDS.shape[1]])
depth["vs shortest"] = depth["ms per written token"] / depth["ms per written token"].iloc[0]
results[f"written token at {int(depth.iloc[-1, 0]):,} vs {int(depth.iloc[0, 0]):,} tokens of chat"] = depth["vs shortest"].iloc[-1]
depth.round(2)

### Plot it: the price of one written token as the chat grows

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(depth["chat length (tokens)"], depth["ms per written token"], "o-")
ax.set_xscale("log", base=2)
ax.set_xlabel("tokens already in the chat")
ax.set_ylabel("ms per written token")
ax.set_ylim(bottom=0)
ax.set_title(f"{MODEL_ID.split('/')[-1]} on {device}")
plt.show()

### The KV cache, by arithmetic

You do not need to run a model to know how big its cache gets. From the config:

`bytes per token = layers x KV heads x head size x 2 (a key and a value) x bytes per number`

We compute it for the loaded model and check it against the cache we just measured. For comparison, the film's Qwen3 8B holds 147,456 bytes per token, about 4.7 GB at 32k tokens, on top of its weights.

In [ ]:
cfg = model.config
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
per_token = cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * 2 * torch.finfo(dtype).bits // 8
print(f"{cfg.num_hidden_layers} layers x {cfg.num_key_value_heads} KV heads x {head_dim} x 2 x "
      f"{torch.finfo(dtype).bits // 8} bytes = {per_token:,} bytes per token")
last = depth.iloc[-1]
n_last = int(last["chat length (tokens)"])
print(f"predicted at {n_last:,} tokens: {per_token * n_last / 1e6:.1f} MB | "
      f"measured: {last['KV cache MB']:.1f} MB")
for n in (1_000, 32_000, 128_000):
    print(f"{n:>8,} tokens of chat -> {per_token * n / 1e9:5.2f} GB read for every written token, "
          f"next to {weight_bytes(model) / 1e9:.2f} GB of weights")

### Step 4: Batching: many answers share one trip through the weights

We ask 1, 4 and 8 **different** questions at the same time. Prompts are padded on the left so every answer starts at the same step, and every answer writes exactly `BATCH_TOKENS` tokens so the counts are comparable.

In [ ]:
QUESTIONS = ["How do plants make food from sunlight?", "Why is the sky blue?",
             "How does a refrigerator work?", "What causes the seasons?",
             "How do vaccines train the immune system?", "Why do cats purr?",
             "How does GPS know where you are?", "What makes bread rise?"]
tok.padding_side = "left"

def chat(question):
    return tok.apply_chat_template([{"role": "user", "content": question + " Answer in about 100 words."}],
                                   add_generation_prompt=True, tokenize=False, enable_thinking=False)

@torch.no_grad()
def answer_batch(n):
    # Answer n different questions at once; return (seconds, tokens written).
    enc = tok([chat(QUESTIONS[i % len(QUESTIONS)]) for i in range(n)], return_tensors="pt", padding=True).to(device)
    sync(); t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=BATCH_TOKENS, min_new_tokens=BATCH_TOKENS,
                         do_sample=False, pad_token_id=tok.pad_token_id)
    sync()
    return time.perf_counter() - t0, n * (out.shape[1] - enc.input_ids.shape[1])

rows = []
for n in BATCH_SIZES:
    seconds, tokens = median_runs(lambda: answer_batch(n))
    rows.append({"answers at once": n, "seconds": seconds, "tokens/s (all answers)": tokens / seconds,
                 "seconds per answer": seconds / n})
batch = pd.DataFrame(rows)
batch["cost per answer vs one at a time"] = batch["seconds per answer"] / batch["seconds per answer"].iloc[0]
results[f"cost per answer, {BATCH_SIZES[-1]} at once vs 1"] = batch["cost per answer vs one at a time"].iloc[-1]
batch.round(2)

The batch takes only a little longer than a single answer, because the expensive part, moving the weights, happens once per step for the whole batch. Total throughput rises and each answer's share of the chip time falls. This is why a busy server answers more cheaply per question than a lone laptop: it always has a batch to fill. With a fully busy chip, time per answer is a fair stand-in for energy per answer; step 5 measures the energy itself.

### Step 5 (optional): Energy from the chip's own power sensor

Time is not energy. To get joules we sample the chip's power while it works, subtract the **idle** power measured just before, and add up the difference over time. This step looks for a sensor and **skips itself if there is none**. Nothing else in the notebook needs it.

- **NVIDIA GPU:** read through NVML (`pip install nvidia-ml-py`). No special permissions.
- **Apple silicon:** macOS only exposes the chip's power to `powermetrics`, which needs administrator rights. The notebook never asks for them. If you want this step, start the meter yourself in a separate terminal, leave it running, and re-run this cell:

```bash
sudo powermetrics --samplers cpu_power,gpu_power -i 250 -o /tmp/pm.txt
```

  The cell then reads the "Combined Power (CPU + GPU + ANE)" lines from that file. Stop the meter with Ctrl-C when you are done.
- **Anything else** (CPU-only laptops, most cloud VMs): skipped.

Close other busy apps first. If something else changes its power draw during the run, the subtraction measures that program instead of the model, so the cell refuses a baseline that is too unsteady and any result that comes out at or below zero.

In [ ]:
import os, re, threading

PM_FILE = "/tmp/pm.txt"

def find_power_meter():
    # Return (name, read_watts) for NVML, or ("powermetrics", None) if the meter file is live, else (None, None).
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        return "nvml", lambda: pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
    except Exception:
        pass
    if os.path.exists(PM_FILE) and time.time() - os.path.getmtime(PM_FILE) < 5:
        return "powermetrics", None
    return None, None

class PowerSampler(threading.Thread):
    # Collect (time, watts) samples in the background from NVML or from powermetrics' output file.
    PM_LINE = re.compile(r"Combined Power \(CPU \+ GPU \+ ANE\):\s*([\d.]+)\s*mW")

    def __init__(self, source, read_watts=None, interval=0.1):
        super().__init__(daemon=True)
        self.source, self.read_watts, self.interval = source, read_watts, interval
        self.samples, self.stopped = [], threading.Event()

    def run(self):
        if self.source == "nvml":
            while not self.stopped.is_set():
                self.samples.append((time.time(), self.read_watts()))
                self.stopped.wait(self.interval)
            return
        with open(PM_FILE, errors="ignore") as f:
            f.seek(0, 2)                       # only samples from now on
            while not self.stopped.is_set():
                line = f.readline()
                if not line:
                    time.sleep(0.05); continue
                m = self.PM_LINE.search(line)
                if m:
                    self.samples.append((time.time(), float(m.group(1)) / 1000.0))

    def watts(self, t0, t1):
        return [w for t, w in self.samples if t0 <= t <= t1]

    def joules(self, t0, t1, idle_w):
        # Energy above idle between t0 and t1 (trapezoid rule).
        s = [(t, w) for t, w in self.samples if t0 <= t <= t1]
        if len(s) < 2:
            return None
        return sum(((wa + wb) / 2 - idle_w) * (tb - ta) for (ta, wa), (tb, wb) in zip(s, s[1:]))

def idle_watts(sampler, seconds=10, max_spread=0.15):
    # Measure idle power; refuse if it moves by more than max_spread (coefficient of variation).
    time.sleep(2)
    t0 = time.time(); time.sleep(seconds)
    w = sampler.watts(t0, time.time())
    if len(w) < 5:
        raise RuntimeError("no power samples arrived; is the meter still running?")
    mean, sd = statistics.mean(w), statistics.pstdev(w)
    if mean > 0 and sd / mean > max_spread and sd > 0.5:
        raise RuntimeError(f"idle power moves by {sd:.2f} W around {mean:.2f} W; something else is busy. "
                           "Close it and re-run this cell.")
    return mean

@torch.no_grad()
def metered(sampler, idle_w, work, min_seconds=6):
    # Repeat work() until min_seconds have passed; return joules above idle and how many tokens it handled.
    tokens = 0
    sync(); t0 = time.time()
    while time.time() - t0 < min_seconds:
        tokens += work(); sync()
    t1 = time.time()
    time.sleep(1.0)                           # let the last samples land
    j = sampler.joules(t0, t1, idle_w)
    if j is None or j <= 0:
        raise RuntimeError(f"measured {j} J above idle, which is impossible: the baseline was disturbed. Re-run.")
    return j, tokens

meter, read_watts = find_power_meter()
if meter is None:
    print("No power sensor found, so the energy step is skipped. Everything above still stands.\n"
          "NVIDIA: pip install nvidia-ml-py.  Apple silicon: start "
          "`sudo powermetrics --samplers cpu_power,gpu_power -i 250 -o /tmp/pm.txt` "
          "in your own terminal, then re-run this cell.")
else:
    sampler = PowerSampler(meter, read_watts); sampler.start()
    prompt = BOOK_IDS[:, :READ_TOKENS]
    read_then_write(model, prompt, 8)          # warm-up

    def one_read():
        model(input_ids=prompt, use_cache=True, logits_to_keep=1)
        return READ_TOKENS

    out = model(input_ids=prompt, use_cache=True, logits_to_keep=1)
    def one_write(n=16):
        write(model, out, n)
        return n

    idle = idle_watts(sampler)
    j_read, n_read = metered(sampler, idle, one_read)
    idle = idle_watts(sampler)
    j_write, n_write = metered(sampler, idle, one_write)
    sampler.stopped.set()
    jpt_read, jpt_write = j_read / n_read, j_write / n_write
    gb = weight_bytes(model) / 1e9
    print(f"meter: {meter} | idle {idle:.2f} W")
    print(f"read : {jpt_read:.5f} J per token ({n_read:,} tokens)")
    print(f"write: {jpt_write:.5f} J per token ({n_write:,} tokens) = {jpt_write / gb:.3f} J per GB of weights")
    print(f"one written token costs {jpt_write / jpt_read:.1f}x the energy of a read token")
    results["J per token read"] = jpt_read
    results["J per token written"] = jpt_write
    results["written / read, energy per token"] = jpt_write / jpt_read

### Results

Everything this run measured on your machine, in one place.

In [ ]:
print(f"{MODEL_ID} on {device}")
pd.DataFrame({"measured": results}).round(3)

## Additional Considerations

### What this means when you use AI

The measurements point at the same few levers the film ends on:

- **Paste in what matters, not everything.** Reading is the cheap phase per token, but a long prompt is still read in full, and it stays in the chat's memory for every token written after it.
- **Ask for only what you need.** Written tokens are the expensive ones, and each one is a full trip through the model. "Answer in three sentences" is a real saving; so is a model that does not write a long visible reasoning trace when you do not need one.
- **Start a fresh chat when the old one stopped mattering.** Step 3 shows every written token getting slower as the conversation grows, because it reads the whole KV cache each time. A new chat resets that cost to zero.
- **Smaller models for small jobs.** Step 2 shows the cost per written token rising with the model's size.
- **Shared servers batch.** Step 4 is why a busy provider answers more cheaply per question than a single laptop. It also means one person's answer on a server is cheaper than the same answer measured alone on a laptop.

### Limits of this measurement

- **Runtime overhead.** Plain PyTorch adds a fixed cost per step that serving runtimes (llama.cpp, MLX, vLLM, TensorRT-LLM) mostly remove. Absolute speeds here are a floor; the ratios are what carry over.
- **Precision.** We load 16-bit weights on a GPU and 32-bit on a CPU. Quantised 4-bit or 8-bit weights move fewer bytes, so they write faster and use less energy per token, which is the same bandwidth story from the other side.
- **The meter's boundary.** NVML reports the GPU board; `powermetrics` reports the chip package (CPU, GPU and neural engine). Neither includes the screen, the fans, the power supply's losses or, for a data center, the cooling.
- **A shared machine lies.** Anything else running changes both time and power. Close other work and run the timing cells twice; if the numbers move a lot, the machine was busy.
- **Attention cost grows with the chat.** At the depths used here the KV cache read is modest next to the weights, so the slowdown in step 3 is gentle for a small model. It grows with longer chats and with models that have more layers or KV heads.

## References

- Pope et al., *Efficiently Scaling Transformer Inference* (MLSys 2023): prefill vs decode, memory-bandwidth-bound generation and batching.
- Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention* (SOSP 2023): the KV cache as the main memory cost of serving.
- Williams, Waterman and Patterson, *Roofline: An Insightful Visual Performance Model for Multicore Architectures* (CACM 2009): compute-bound vs memory-bound.
- Luccioni, Jernite and Strubell, *Power Hungry Processing: Watts Driving the Cost of AI Deployment?* (FAccT 2024): measured energy of model inference.
- Qwen Team, *Qwen3 Technical Report* (2025), [models on Hugging Face](https://huggingface.co/Qwen).
- Arthur Conan Doyle, *The Adventures of Sherlock Holmes*, [Project Gutenberg #1661](https://www.gutenberg.org/ebooks/1661) (public domain).
- [NVIDIA Management Library (NVML)](https://developer.nvidia.com/management-library-nvml) and Apple's `powermetrics` manual page (`man powermetrics`).